In [3]:
!pip install uv
!uv pip install maxtext[cuda12]==0.2.1 --system --resolution=lowest

Using Python 3.12.13 environment at: /usr
Resolved 260 packages in 90ms
Prepared 1 package in 2m 14s
Uninstalled 36 packages in 276ms
Installed 98 packages in 131ms
 - absl-py==1.4.0
 + absl-py==2.3.1
 - aiofiles==24.1.0
 + aiofiles==25.1.0
 + aqtp==0.9.0
 + astroid==4.0.2
 + auditwheel==6.5.0
 + black==24.10.0
 + build==1.3.0
 + cfgv==3.5.0
 + cheroot==11.1.2
 + chex==0.1.91
 + cloud-accelerator-diagnostics==0.1.1
 + cloud-tpu-diagnostics==0.1.5
 + clu==0.0.12
 + colorama==0.4.6
 + coverage==7.12.0
 - datasets==4.0.0
 + datasets==4.4.1
 - decorator==4.4.2
 + decorator==5.2.1
 - dill==0.3.8
 + dill==0.4.0
 + distlib==0.4.0
 + drjax==0.1.4
 + einshape==1.0
 + evaluate==0.4.6
 + execnet==2.1.2
 - flax==0.11.2
 + flax==0.12.1
 - fsspec==2025.3.0
 + fsspec==2025.10.0
 - gcsfs==2025.3.0
 + gcsfs==2025.10.0
 + google-cloud-mldiagnostics==0.5.10
 + gviz-api==1.10.0
 + hypothesis==6.142.1
 + identify==2.6.15
 + importlab==0.8.1
 + isort==7.0.0
 - jax==0.7.2
 + jax==0.8.1
 - jax-cuda12-pjrt==0.

In [4]:
ls /usr/local/lib/python3.12/dist-packages/maxtext/configs/models/ | grep deepseek

deepseek2-16b.yml
deepseek2-236b.yml
deepseek3.2-671b.yml
deepseek3-671b-2dfsdp.yml
deepseek3-671b.yml
deepseek3-test.yml
deepseek3-tiny.yml
deepseek-custom.yml


In [5]:
!find /usr/local/lib/python3.12/dist-packages/maxtext -iname "*deepseek*"

/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/deepseek3-tiny.yml
/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/deepseek-custom.yml
/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/deepseek3-671b.yml
/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/deepseek2-236b.yml
/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/deepseek3.2-671b.yml
/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/deepseek2-16b.yml
/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/deepseek3-671b-2dfsdp.yml
/usr/local/lib/python3.12/dist-packages/maxtext/configs/models/deepseek3-test.yml
/usr/local/lib/python3.12/dist-packages/maxtext/checkpoint_conversion/standalone_scripts/convert_deepseek_family_unscanned_ckpt.py
/usr/local/lib/python3.12/dist-packages/maxtext/checkpoint_conversion/standalone_scripts/deepseek_fp8_to_bf16.py
/usr/local/lib/python3.12/dist-packages/maxtext/checkpoint_conversion/standalone_scripts/co

In [6]:
!cat /usr/local/lib/python3.12/dist-packages/maxtext/configs/models/deepseek3-tiny.yml

# Copyright 2023–2025 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#    https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Tiny version of DeepSeek V3 for testing.

base_emb_dim: 64
base_num_query_heads: 4
base_num_kv_heads: 4
base_mlp_dim: 64
base_moe_mlp_dim: 64
base_num_decoder_layers: 61
first_num_dense_layers: 3
mlp_activations: ["silu","linear"]
vocab_size: 129280
enable_dropout: False
logits_via_embedding: False
normalization_layer_epsilon: 1.0e-6
num_experts: 16
num_experts_per_tok: 8
shared_experts: 1
routed_scaling_factor: 2.

In [7]:
!nvidia-smi

Sat Jun 20 07:18:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   36C    P0             56W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [7]:
!TF_GPU_ALLOCATOR=cuda_malloc_async python -m maxtext.trainers.pre_train.train \
  /usr/local/lib/python3.12/dist-packages/maxtext/configs/base.yml \
  model_name=deepseek3-tiny \
  dataset_type=synthetic \
  steps=50 \
  run_name=deepseek3_tiny_moe_gpu_50steps \
  base_output_directory=/content/maxtext_outputs \
  skip_jax_distributed_system=True \
  checkpoint_period=100000 \
  log_period=1 \
  per_device_batch_size=1 \
  global_batch_size_to_train_on=1 \
  global_batch_size_to_load=1 \
  global_batch_size_to_eval_on=1 \
  max_target_length=512

W0620 00:30:40.183774 134627084448384 pyconfig.py:211] tokenizer_path not found in HF_IDS in maxtext/src/maxtext/utils/globals.py.           Using the default src/maxtext/assets/tokenizers/tokenizer.llama2 instead.           Please pass tokenizer_path in your command if this is not intended.
I0620 00:30:40.184144 134627084448384 max_utils.py:197] Skipping jax distributed system due to skip_jax_distributed_system=True flag.
INFO:2026-06-20 00:30:40,278:jax._src.xla_bridge:812: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
I0620 00:30:40.278696 134627084448384 xla_bridge.py:812] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
W0620 00:30:40.552061    8061 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the 

This error is from DeepSeek MLA attention + GPU Pallas attention:

q head dim = 192
k head dim = 192
v head dim = 128

The GPU Pallas kernel expects q, k, v to have the same head dimension, but DeepSeek MLA has different value head dimension.

Try forcing a non-Pallas attention path:

If that still errors, we will make a small custom config where:

v_head_dim: 192

so q/k/v dimensions match for the GPU kernel.

In [8]:
!TF_GPU_ALLOCATOR=cuda_malloc_async python -m maxtext.trainers.pre_train.train \
  /usr/local/lib/python3.12/dist-packages/maxtext/configs/base.yml \
  model_name=deepseek3-tiny \
  dataset_type=synthetic \
  steps=50 \
  run_name=deepseek3_tiny_moe_gpu_50steps_dot \
  base_output_directory=/content/maxtext_outputs \
  skip_jax_distributed_system=True \
  checkpoint_period=100000 \
  log_period=1 \
  per_device_batch_size=1 \
  global_batch_size_to_train_on=1 \
  global_batch_size_to_load=1 \
  global_batch_size_to_eval_on=1 \
  max_target_length=512 \
  attention=dot_product

W0620 00:32:21.317441 136706604110464 pyconfig.py:211] tokenizer_path not found in HF_IDS in maxtext/src/maxtext/utils/globals.py.           Using the default src/maxtext/assets/tokenizers/tokenizer.llama2 instead.           Please pass tokenizer_path in your command if this is not intended.
I0620 00:32:21.317757 136706604110464 max_utils.py:197] Skipping jax distributed system due to skip_jax_distributed_system=True flag.
INFO:2026-06-20 00:32:21,387:jax._src.xla_bridge:812: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
I0620 00:32:21.387997 136706604110464 xla_bridge.py:812] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
W0620 00:32:21.633649    8602 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the 

In [11]:
!cat /usr/local/lib/python3.12/dist-packages/maxtext/configs/models/deepseek-custom.yml

# Copyright 2023–2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#    https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Small model config for testing (derived from DeepSeek V3.2 - 671B)
# Included modules: DeepSeek Sparse Attention, Engram, mHC

# Example command:
# python3 -m maxtext.trainers.pre_train.train src/maxtext/configs/base.yml base_output_directory=${BASE_OUTPUT_PATH} run_name=demo model_name=deepseek-custom scan_layers=True attention=flash use_tokamax_splash=True enable_checkpointing=false async_checkpointing=false data

In [16]:
!TF_GPU_ALLOCATOR=cuda_malloc_async python -m maxtext.trainers.pre_train.train \
  /usr/local/lib/python3.12/dist-packages/maxtext/configs/base.yml \
  model_name=deepseek-custom \
  dataset_type=synthetic \
  steps=50 \
  run_name=deepseek_custom_moe_gpu_50steps_no_engram \
  base_output_directory=/content/maxtext_outputs \
  skip_jax_distributed_system=True \
  checkpoint_period=100000 \
  log_period=1 \
  per_device_batch_size=1 \
  global_batch_size_to_train_on=1 \
  global_batch_size_to_load=1 \
  global_batch_size_to_eval_on=1 \
  max_target_length=512 \
  attention=flash \
  use_tokamax_splash=True \
  enable_checkpointing=false \
  async_checkpointing=false \
  engram_layers=[] \
  use_indexer=False

W0620 00:41:50.931436 140056135873152 pyconfig.py:211] tokenizer_path not found in HF_IDS in maxtext/src/maxtext/utils/globals.py.           Using the default src/maxtext/assets/tokenizers/tokenizer.llama2 instead.           Please pass tokenizer_path in your command if this is not intended.
I0620 00:41:50.931752 140056135873152 max_utils.py:197] Skipping jax distributed system due to skip_jax_distributed_system=True flag.
INFO:2026-06-20 00:41:50,999:jax._src.xla_bridge:812: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
I0620 00:41:50.999140 140056135873152 xla_bridge.py:812] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
W0620 00:41:51.242862   11581 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the 

In [17]:
!TF_GPU_ALLOCATOR=cuda_malloc_async python -m maxtext.trainers.pre_train.train \
  /usr/local/lib/python3.12/dist-packages/maxtext/configs/base.yml \
  model_name=deepseek-custom \
  dataset_type=synthetic \
  steps=50 \
  run_name=deepseek_custom_moe_gpu_50steps_v48 \
  base_output_directory=/content/maxtext_outputs \
  skip_jax_distributed_system=True \
  checkpoint_period=100000 \
  log_period=1 \
  per_device_batch_size=1 \
  global_batch_size_to_train_on=1 \
  global_batch_size_to_load=1 \
  global_batch_size_to_eval_on=1 \
  max_target_length=512 \
  attention=flash \
  use_tokamax_splash=True \
  enable_checkpointing=false \
  async_checkpointing=false \
  engram_layers=[] \
  use_indexer=False \
  v_head_dim=48

W0620 00:43:31.727172 136473573274240 pyconfig.py:211] tokenizer_path not found in HF_IDS in maxtext/src/maxtext/utils/globals.py.           Using the default src/maxtext/assets/tokenizers/tokenizer.llama2 instead.           Please pass tokenizer_path in your command if this is not intended.
I0620 00:43:31.727509 136473573274240 max_utils.py:197] Skipping jax distributed system due to skip_jax_distributed_system=True flag.
INFO:2026-06-20 00:43:31,798:jax._src.xla_bridge:812: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
I0620 00:43:31.798025 136473573274240 xla_bridge.py:812] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
W0620 00:43:32.044009   12112 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the 

In [19]:
!pip install -U protobuf==6.31.1

In [20]:
!TF_GPU_ALLOCATOR=cuda_malloc_async python -m maxtext.trainers.pre_train.train \
  /usr/local/lib/python3.12/dist-packages/maxtext/configs/base.yml \
  model_name=deepseek-custom \
  dataset_type=synthetic \
  steps=50 \
  run_name=deepseek_custom_moe_gpu_50steps_dot \
  base_output_directory=/content/maxtext_outputs \
  skip_jax_distributed_system=True \
  checkpoint_period=100000 \
  log_period=1 \
  per_device_batch_size=1 \
  global_batch_size_to_train_on=1 \
  global_batch_size_to_load=1 \
  global_batch_size_to_eval_on=1 \
  max_target_length=512 \
  attention=dot_product \
  use_tokamax_splash=False \
  enable_checkpointing=false \
  async_checkpointing=false \
  engram_layers=[] \
  use_indexer=False \
  v_head_dim=128

/usr/local/lib/python3.12/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/attr_value.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/tensor.proto. Please update the gencode to avoid compatibility violations in the next runtime release.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/google/protobuf/runtime_version.py:98: UserWarning: Protobuf gencode version 5.28.3 is exactly one major version older than the runtime version 6.31.1 at tensorflow/core/framework/resource_handle.proto. Please update the gencode to avoid compatibility violations in the next r

In [9]:
!TF_GPU_ALLOCATOR=cuda_malloc_async python -m maxtext.trainers.pre_train.train \
  /usr/local/lib/python3.12/dist-packages/maxtext/configs/base.yml \
  model_name=deepseek-custom \
  hardware=gpu \
  dataset_type=synthetic \
  steps=5 \
  run_name=deepseek_custom_debug_dense_gpu \
  base_output_directory=/content/maxtext_outputs \
  skip_jax_distributed_system=True \
  checkpoint_period=100000 \
  log_period=1 \
  per_device_batch_size=1 \
  global_batch_size_to_train_on=1 \
  global_batch_size_to_load=1 \
  global_batch_size_to_eval_on=1 \
  max_target_length=512 \
  attention=dot_product \
  use_tokamax_splash=False \
  enable_checkpointing=false \
  async_checkpointing=false \
  engram_layers=[] \
  use_indexer=False \
  megablox=False \
  sparse_matmul=False \
  num_experts=1 \
  num_experts_per_tok=1 \
  shared_experts=1

W0620 07:21:29.771698 137898457477760 pyconfig.py:211] tokenizer_path not found in HF_IDS in maxtext/src/maxtext/utils/globals.py.           Using the default src/maxtext/assets/tokenizers/tokenizer.llama2 instead.           Please pass tokenizer_path in your command if this is not intended.
I0620 07:21:29.772037 137898457477760 max_utils.py:197] Skipping jax distributed system due to skip_jax_distributed_system=True flag.
INFO:2026-06-20 07:21:29,861:jax._src.xla_bridge:812: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
I0620 07:21:29.861381 137898457477760 xla_bridge.py:812] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
W0620 07:21:30.129868   31076 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the 

In [11]:
!python -c "import jax; print(jax.__version__)"
!python -c "import jaxlib; print(jaxlib.__version__)"
!pip show maxtext

0.8.1
0.8.1
Name: maxtext
Version: 0.2.1
Summary: MaxText is a simple, performant and scalable Jax LLM!
Home-page: 
Author: 
Author-email: 
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: 
Required-by: 


In [12]:
!TF_GPU_ALLOCATOR=cuda_malloc_async python -m maxtext.trainers.pre_train.train \
  /usr/local/lib/python3.12/dist-packages/maxtext/configs/base.yml \
  model_name=deepseek-custom \
  hardware=gpu \
  dataset_type=synthetic \
  steps=50 \
  run_name=deepseek_custom_moe_gpu_no_megablox \
  base_output_directory=/content/maxtext_outputs \
  skip_jax_distributed_system=True \
  checkpoint_period=100000 \
  log_period=1 \
  per_device_batch_size=1 \
  global_batch_size_to_train_on=1 \
  global_batch_size_to_load=1 \
  global_batch_size_to_eval_on=1 \
  max_target_length=512 \
  attention=dot_product \
  use_tokamax_splash=False \
  enable_checkpointing=false \
  async_checkpointing=false \
  engram_layers=[] \
  use_indexer=False \
  megablox=False \
  sparse_matmul=False \
  num_experts=16 \
  num_experts_per_tok=2 \
  shared_experts=1

W0620 07:29:47.565912 139412573323904 pyconfig.py:211] tokenizer_path not found in HF_IDS in maxtext/src/maxtext/utils/globals.py.           Using the default src/maxtext/assets/tokenizers/tokenizer.llama2 instead.           Please pass tokenizer_path in your command if this is not intended.
I0620 07:29:47.566237 139412573323904 max_utils.py:197] Skipping jax distributed system due to skip_jax_distributed_system=True flag.
INFO:2026-06-20 07:29:47,638:jax._src.xla_bridge:812: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
I0620 07:29:47.638738 139412573323904 xla_bridge.py:812] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
W0620 07:29:47.890675   34533 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not used by the 

In [13]:
!mkdir -p /content/task3_saved_logs/deepseek_custom_moe_gpu

In [14]:
%%writefile /content/task3_saved_logs/deepseek_custom_moe_gpu/config_used.txt
model_name=deepseek-custom
hardware=gpu
dataset_type=synthetic
steps=50
run_name=deepseek_custom_moe_gpu_no_megablox
max_target_length=512
per_device_batch_size=1
num_experts=16
num_experts_per_tok=2
shared_experts=1
megablox=False
sparse_matmul=False
attention=dot_product
use_tokamax_splash=False
engram_layers=[]
use_indexer=False
enable_checkpointing=false

Writing /content/task3_saved_logs/deepseek_custom_moe_gpu/config_used.txt


In [15]:
%%writefile /content/task3_saved_logs/deepseek_custom_moe_gpu/final_metrics.txt
completed step: 49
seconds: 0.040
TFLOP/s/device: 14.464
Tokens/s/device: 12944.329
total_weights: 512
loss: 2.083
number parameters: 0.308 billion
GPU: NVIDIA A100-SXM4-80GB
JAX: 0.8.1
JAXLIB: 0.8.1
MaxText: 0.2.1

Writing /content/task3_saved_logs/deepseek_custom_moe_gpu/final_metrics.txt


In [16]:
!cp -r /content/maxtext_outputs/deepseek_custom_moe_gpu_no_megablox/tensorboard \
/content/task3_saved_logs/deepseek_custom_moe_gpu/

In [17]:
!zip -r task3_deepseek_gpu_logs.zip /content/task3_saved_logs

  adding: content/task3_saved_logs/ (stored 0%)
  adding: content/task3_saved_logs/deepseek_custom_moe_gpu/ (stored 0%)
  adding: content/task3_saved_logs/deepseek_custom_moe_gpu/config_used.txt (deflated 39%)
  adding: content/task3_saved_logs/deepseek_custom_moe_gpu/final_metrics.txt (deflated 19%)
  adding: content/task3_saved_logs/deepseek_custom_moe_gpu/tensorboard/ (stored 0%)
  adding: content/task3_saved_logs/deepseek_custom_moe_gpu/tensorboard/deepseek_custom_moe_gpu_no_megablox/ (stored 0%)
  adding: content/task3_saved_logs/deepseek_custom_moe_gpu/tensorboard/deepseek_custom_moe_gpu_no_megablox/events.out.tfevents.1781940671.f61a52497f7b (deflated 75%)
